# 公众号 / 头条号 配图生成工作台（Colab + T4）· 升级版

**能做什么**：输入文章标题+正文 → **LLM 自动把正文提炼成配图提示词** → 自动生图 → 上传 **GitHub 图床**拿外链 → **一键真正发文**到公众号 / 头条号。

**本次升级**
1. 🧠 **LLM 提炼提示词**：正文不再只靠标题，而是由大模型浓缩成贴合画面的英文提示词（API 优先，本地小模型兜底，无 key 自动回退标题）。
2. 🚀 **真·一键发文**：公众号走「上传素材 → 建草稿 → 发布」完整链路；头条号走「取 token → 传图 → 提交」。

**为什么适合你的 10G 网盘**：默认不挂载网盘；模型只下到临时盘；成品 `files.download` 落本机。
> ⚠️ 保密：此流程用于**公开发布**内容。当事人材料/合同/证据等涉密内容请勿上传，回你本地 P4 离线处理。

## 0. 配置区（先改这里）

**图床**：Colab 左侧 🔑 Secrets 新增 `GH_TOKEN`（GitHub 令牌，需 `repo` 权限）；建一个**公开**仓库当图床。
**LLM（提炼提示词）**：Secrets 加 `LLM_API_KEY`、`LLM_BASE_URL`(可选, 默认 DeepSeek)、`LLM_MODEL`(可选)。没有 key 也能跑（回退标题 / 本地小模型）。
**发文**：Secrets 加 `WX_APPID`+`WX_SECRET`（公众号）、`TT_KEY`+`TT_SECRET`（头条号）。
`DRY_RUN=True` 表示**只模拟、不真正推送**；确认无误后改成 `False` 才真正发布。

In [ ]:
# ===== 0. 配置区（按需修改）=====
# GitHub 图床
GITHUB_REPO   = "你的用户名/你的图床仓库"   # 例: "lawyerx/img-bed"
GITHUB_BRANCH = "main"
IMAGE_FOLDER  = "images"

# 生图方式: 'online' 在线零下载(默认) | 'sd' 本地SD(需GPU)
GEN_MODE = "online"

# LLM 提炼提示词: 'auto'(有key用API,否则本地,再否则标题) | 'api' | 'local' | 'none'
LLM_MODE = "auto"

# 安全开关：True=只模拟发文不真正推送；False=真正发布
DRY_RUN = True

# 图片风格预设（生图模型更吃英文）
STYLES = {
    "新闻插画": "editorial news illustration, clean flat vector style, Chinese newspaper aesthetic",
    "水彩":     "soft watercolor painting, warm tones, artistic",
    "科技蓝":   "futuristic tech style, blue tone, isometric, 3d render",
    "极简":     "minimalist flat design, pastel colors, lots of whitespace",
    "国风":     "Chinese traditional ink painting style, elegant, muted colors",
}

# 从 Colab Secrets 安全读取（不暴露在代码里）
from google.colab import userdata
def sec(k, default=""):
    try: return userdata.get(k) or default
    except Exception: return default

GH_TOKEN  = sec('GH_TOKEN')
LLM_API_KEY = sec('LLM_API_KEY')
LLM_BASE_URL = sec('LLM_BASE_URL', "https://api.deepseek.com/v1")
LLM_MODEL   = sec('LLM_MODEL', "deepseek-chat")
WX_APPID  = sec('WX_APPID');  WX_SECRET = sec('WX_SECRET')
TT_KEY    = sec('TT_KEY');    TT_SECRET = sec('TT_SECRET')

print("==== 环境检查 ====")
import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print("GH_TOKEN:", bool(GH_TOKEN), "| LLM_API_KEY:", bool(LLM_API_KEY), "| LLM_MODE:", LLM_MODE)
print("WX:", bool(WX_APPID and WX_SECRET), "| TT:", bool(TT_KEY and TT_SECRET), "| DRY_RUN:", DRY_RUN)

## 1. 基础提示词（无 LLM 时的兜底）

把标题直接变成英文画面描述，作为 LLM 不可用时的回退方案。

In [ ]:
def build_prompt(article_title, article_text="", style="新闻插画", extra=""):
    """标题直出提示词（LLM 不可用时的兜底）。"""
    base = STYLES.get(style, STYLES["新闻插画"])
    subject = article_title.strip().replace("\n", " ")
    prompt = f"{base}, depicting: {subject}"
    if extra:
        prompt += f", {extra}"
    prompt += ", high quality, clear composition, no watermark, no text overlay"
    return prompt

print(build_prompt("民法典新规：高空抛物谁来担责", style="新闻插画"))

## 2A. 方式A：在线生图（默认）

pollinations/flux 免费接口，零下载、不吃显存、秒出。

In [ ]:
import requests, os
from io import BytesIO
from PIL import Image
from datetime import datetime

OUT_DIR = "/content/output"
os.makedirs(OUT_DIR, exist_ok=True)

def generate_image_online(prompt, width=1024, height=576, seed=None):
    """在线免费生图，返回 PIL 图片。"""
    import urllib.parse
    p = urllib.parse.quote(prompt)
    url = f"https://image.pollinations.ai/prompt/{p}?width={width}&height={height}&nologo=true&model=flux"
    if seed is not None:
        url += f"&seed={seed}"
    r = requests.get(url, timeout=180)
    r.raise_for_status()
    return Image.open(BytesIO(r.content)).convert("RGB")

img = generate_image_online(build_prompt("民法典新规：高空抛物谁来担责", style="新闻插画"))
fname = f"{OUT_DIR}/cover_{datetime.now().strftime('%H%M%S')}.png"
img.save(fname)
print("已生成：", fname, img.size)
img

## 2B. 方式B：本地 Stable Diffusion（可选，质量更高）

前提：运行时 → 更改运行时类型 → 选 **GPU**（T4）。模型下到临时盘 `/content`，不占 10G 网盘。

In [ ]:
# 仅方式B需要：安装 diffusers
# !pip install -q diffusers transformers accelerate

def generate_image_sd(prompt, steps=25):
    """本地 SDXL 生图，模型缓存于 /content 临时盘。"""
    import torch
    from diffusers import StableDiffusionXLPipeline
    pipe = StableDiffusionXLPipeline.from_pretrained(
        "stabilityai/stable-diffusion-xl-base-1.0",
        torch_dtype=torch.float16, use_safetensors=True, variant="fp16").to("cuda")
    return pipe(prompt, num_inference_steps=steps).images[0]

if GEN_MODE == "sd":
    img = generate_image_sd(build_prompt("民法典新规：高空抛物谁来担责", style="新闻插画"))
    fname = f"{OUT_DIR}/cover_sd.png"; img.save(fname); print("SD 已生成：", fname)
else:
    print("当前 online 模式。需要更高画质时把 GEN_MODE 改为 'sd' 并开启 GPU。")

## 2C. 🧠 LLM 提炼提示词（升级核心）

把**正文**交给大模型，浓缩成一句贴合画面的英文生图提示词。
- `LLM_MODE='auto'`（默认）：有 `LLM_API_KEY` 走 API（质量最好），否则尝试本地小模型，再否则回退标题。
- 本地小模型用 Qwen2.5-1.5B（首次下载约 3GB 到临时盘，仅本地模式需要）。

In [ ]:
# 仅本地LLM需要：取消下一行注释
# !pip install -q transformers accelerate

def llm_distill_api(title, text, style="新闻插画"):
    sys_p = "You are an image-prompt engineer. Given a Chinese article title and body, output ONE concise English image-generation prompt. No explanation, no quotes."
    user_p = f"Title: {title}\nBody: {text[:2000]}\nStyle keywords: {STYLES.get(style, style)}\nOutput only the English prompt:"
    r = requests.post(LLM_BASE_URL + "/chat/completions",
        headers={"Authorization": f"Bearer {LLM_API_KEY}"},
        json={"model": LLM_MODEL, "temperature": 0.7,
              "messages":[{"role":"system","content":sys_p},{"role":"user","content":user_p}]},
        timeout=60)
    return r.json()["choices"][0]["message"]["content"].strip().strip('"') + ", high quality, no watermark, no text"

def llm_distill_local(title, text, style="新闻插画"):
    import os
    # 模型缓存落到临时盘 /content，不占 10G 网盘；用环境变量而非 pipeline 的 cache_dir 参数
    os.environ.setdefault("HF_HOME", "/content/models")
    os.environ.setdefault("TRANSFORMERS_CACHE", "/content/models")
    from transformers import pipeline
    import torch
    mdl = "Qwen/Qwen2.5-1.5B-Instruct"
    print("加载本地小模型", mdl, "(首次下载到临时盘 /content/models, 约3GB)")
    gen = pipeline("text-generation", model=mdl,
                   torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
                   device_map="auto")
    out = gen([{"role":"system","content":"Output ONE English image prompt for the article. No explanation."},
               {"role":"user","content":f"Title:{title}\nBody:{text[:1500]}\nStyle:{STYLES.get(style,style)}\nPrompt:"}],
              max_new_tokens=120, do_sample=False)
    out = out[0]["generated_text"][-1]["content"]
    return out.strip().strip('"') + ", high quality, no watermark, no text"

def llm_distill_prompt(title, text, style="新闻插画"):
    """自动选后端：API → 本地 → 标题兜底。"""
    if not text.strip():
        return build_prompt(title, text, style)
    if LLM_MODE in ("auto", "api") and LLM_API_KEY:
        try: return llm_distill_api(title, text, style)
        except Exception as e: print("⚠️ API 提炼失败, 回退:", e)
    if LLM_MODE in ("auto", "local"):
        try: return llm_distill_local(title, text, style)
        except Exception as e: print("⚠️ 本地模型失败, 回退标题:", e)
    return build_prompt(title, text, style)

# 试试：给一段正文，看 LLM 提炼出的提示词
demo_text = "高空抛物被称为'悬在城市上空的痛'。新规明确，从建筑物中抛掷物品造成他人损害的，由侵权人依法承担侵权责任；经调查难以确定具体侵权人的，除能够证明自己不是侵权人的外，由可能加害的建筑物使用人给予补偿。"
print(llm_distill_prompt("民法典新规：高空抛物谁来担责", demo_text, style="新闻插画"))

## 3. 一篇文章生成多张配图（接入 LLM）

`use_llm=True` 时，每张图都基于**正文提炼**的提示词生成。

In [ ]:
def generate_for_article(title, text="", n=3, style="新闻插画", mode=None, use_llm=False):
    """生成 n 张配图，返回 (图片列表, 文件名列表)。"""
    mode = mode or GEN_MODE
    imgs, names = [], []
    for i in range(n):
        pr = llm_distill_prompt(title, text, style) if use_llm else build_prompt(title, text, style)
        pr = pr + f", variant {i+1}"
        im = generate_image_sd(pr) if mode == "sd" else generate_image_online(pr, seed=1000 + i)
        nm = f"{OUT_DIR}/{title[:20].strip()}_{i+1}.png"
        im.save(nm); imgs.append(im); names.append(nm)
    return imgs, names

# 示例（用 LLM 提炼）
imgs, names = generate_for_article(
    "个人信息保护法对企业的影响",
    text="企业收集员工和客户信息必须明示同意，违规最高可处营业额5%罚款。",
    n=2, style="科技蓝", use_llm=True)
print("生成完成：", names)

## 4. 上传 GitHub 图床，拿外链

返回 **jsDelivr** 外链（国内相对快）与 raw 链接。仓库须**公开**。

In [ ]:
import base64

def upload_to_github(image_path, repo=GITHUB_REPO, branch=GITHUB_BRANCH, folder=IMAGE_FOLDER):
    if not GH_TOKEN:
        raise ValueError("请先在 Colab Secrets 设置 GH_TOKEN")
    with open(image_path, "rb") as f:
        content = base64.b64encode(f.read()).decode()
    fname = os.path.basename(image_path)
    path = f"{folder}/{fname}"
    api = f"https://api.github.com/repos/{repo}/contents/{path}"
    r = requests.put(api,
        headers={"Authorization": f"token {GH_TOKEN}", "Accept": "application/vnd.github+json"},
        json={"message": f"add {fname}", "content": content, "branch": branch})
    r.raise_for_status()
    d = r.json()["content"]
    return {"raw": d["download_url"], "jsdelivr": f"https://cdn.jsdelivr.net/gh/{repo}@{branch}/{path}"}

url_info = upload_to_github(names[0])
print("jsDelivr:", url_info["jsdelivr"])
print("raw     :", url_info["raw"])

## 5. 🚀 多平台发文（完整可调用）

- **公众号**：`上传图片素材 → 建草稿(draft/add) → 发布(freepublish/submit)`，返回发布结果。
- **头条号**：`取 token → 传图 → 提交内容(content/submit)`。endpoint 以你开放平台后台最新文档为准（已在常量里标注，便于修改）。
- 密钥全部走 Secrets，不写死在代码。

In [ ]:
import time

# ---------- 公众号 ----------
_wx_token = {"token": None, "exp": 0}
def wx_get_token(appid, appsecret):
    if _wx_token["token"] and time.time() < _wx_token["exp"]:
        return _wx_token["token"]
    r = requests.get(
        f"https://api.weixin.qq.com/cgi-bin/token?grant_type=client_credential&appid={appid}&secret={appsecret}",
        timeout=20).json()
    _wx_token["token"] = r["access_token"]; _wx_token["exp"] = time.time() + r["expires_in"] - 200
    return r["access_token"]

def wechat_publish(title, content_html, image_paths, appid, appsecret, author="", dry_run=True):
    """完整发布：上传图片 → 建草稿 → (发布)。dry_run=True 只建草稿不发布。"""
    token = wx_get_token(appid, appsecret)
    media_ids, urls = [], []
    for p in image_paths:
        r = requests.post(
            f"https://api.weixin.qq.com/cgi-bin/material/add_material?access_token={token}&type=image",
            files={"media": open(p, "rb")}, timeout=60).json()
        media_ids.append(r.get("media_id")); urls.append(r.get("url"))
    article = {"title": title, "author": author, "content": content_html,
               "thumb_media_id": media_ids[0] or "", "digest": "", "content_source_url": "",
               "need_open_comment": 0, "only_fans_can_comment": 0}
    add = requests.post(f"https://api.weixin.qq.com/cgi-bin/draft/add?access_token={token}",
                        json={"articles": [article]}, timeout=60).json()
    draft_id = add.get("media_id")
    if dry_run:
        return {"dry_run": True, "article_urls": urls, "draft_media_id": draft_id}
    pub = requests.post(f"https://api.weixin.qq.com/cgi-bin/freepublish/submit?access_token={token}",
                        json={"media_id": draft_id}, timeout=60).json()
    return {"article_urls": urls, "draft_media_id": draft_id, "publish": pub}

# ---------- 头条号 ----------
TOUTIAO_TOKEN_URL   = "https://open.snssdk.com/oauth2/access_token/"
TOUTIAO_PUBLISH_URL = "https://open.snssdk.com/content/submit/"   # 请在开放平台核对最新 endpoint
TOUTIAO_IMG_URL     = "https://open.snssdk.com/image/upload/"

def toutiao_get_token(key, secret):
    r = requests.get(TOUTIAO_TOKEN_URL,
        params={"client_key": key, "client_secret": secret, "grant_type": "client_credentials"},
        timeout=20).json()
    return r.get("access_token")

def toutiao_publish(title, content_html, image_paths, client_key, client_secret, dry_run=True):
    """头条号发布骨架：取 token → 传图 → 提交。dry_run=True 只构造不提交。"""
    token = toutiao_get_token(client_key, client_secret)
    uris = []
    for p in image_paths:
        up = requests.post(TOUTIAO_IMG_URL, params={"access_token": token},
                           files={"image": open(p, "rb")}, timeout=60).json()
        uris.append((up.get("web_uri") or up.get("data", {}) or {}).get("web_uri") if isinstance(up.get("data"), dict) else up.get("web_uri"))
    payload = {"title": title, "content": content_html, "cover_images": [u for u in uris if u]}
    if dry_run:
        return {"dry_run": True, "token_ok": bool(token), "payload": payload}
    r = requests.post(TOUTIAO_PUBLISH_URL, params={"access_token": token},
                      json=payload, timeout=60).json()
    return r

print("发文模块已加载：wechat_publish() / toutiao_publish() 可用。")

## 6. 一键全流程（LLM 提炼 + 预览 + 确认发布）

给标题+正文 → LLM 提炼提示词 → 生图 → 传图床 → **渲染图文 HTML 预览**。
**不会自动发布**：请到 6.5 检查预览，把 `CONFIRM_PUBLISH` 改为 `True` 后，到 6.6 才真正推送；受 `DRY_RUN` 控制。

In [ ]:
def build_article_html(title, urls, text, author=""):
    """生成可直接预览/发布的图文 HTML（含内联样式）。"""
    imgs = "".join(
        f'<p style="text-align:center;margin:8px 0"><img src="{u}" '
        f'style="max-width:100%;border-radius:6px"></p>' for u in urls)
    body = (text or "").replace("\n", "<br>")
    meta = f'<div style="color:#999;font-size:13px;margin:4px 0 16px">作者：{author}</div>' if author else ""
    return f"""<!doctype html><html lang="zh"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1"><title>{title}</title></head>
<body style="font-family:-apple-system,'PingFang SC','Microsoft YaHei',sans-serif;max-width:680px;margin:0 auto;padding:24px;color:#1a1a1a;line-height:1.8">
<h1 style="font-size:24px;line-height:1.4;margin:0 0 6px">{title}</h1>
{meta}
{imgs}
<div style="margin-top:14px">{body}</div>
</body></html>"""

def preview_article(title, urls, text, author=""):
    """渲染图文预览并保存 preview.html（不发布）。"""
    html = build_article_html(title, urls, text, author)
    try:
        from IPython.display import display, HTML
        display(HTML(html))
    except Exception:
        print("（无法内联预览，已保存 preview.html 供下载查看）")
    with open("/content/preview.html", "w", encoding="utf-8") as f:
        f.write(html)
    return html

# 流程结果暂存（供『6.5 预览并确认发布』步骤使用）
LAST = {}

def run_pipeline(title, text="", n=3, style="新闻插画", use_llm=True, author=""):
    print("① 生成配图…")
    imgs, names = generate_for_article(title, text, n=n, style=style, use_llm=use_llm)
    print("② 上传 GitHub 图床…")
    urls = [upload_to_github(nm)["jsdelivr"] for nm in names]
    for u in urls: print(" -", u)
    print("③ 渲染图文预览…")
    html = preview_article(title, urls, text, author)
    global LAST
    LAST = {"title": title, "text": text, "urls": urls,
            "names": names, "html": html, "author": author}
    print("✅ 配图与预览已就绪。请到下面『6.5 预览并确认发布』检查预览，确认无误后再发布。")
    return urls

# 示例：一篇"离婚冷静期"普法文章
urls = run_pipeline(
    "离婚冷静期：你该知道的3个要点",
    text="民法典规定，自婚姻登记机关收到离婚登记申请之日起三十日内，任何一方可撤回申请。",
    n=2, style="新闻插画", use_llm=True, author="方晓晖")

## 6.5 预览并确认发布

上面已经：① 生成配图 → ② 上传图床 → ③ 渲染图文预览（也已保存 `/content/preview.html` 可下载查看）。

**请先仔细检查预览里的图片、标题、正文是否无误。**
确认无误后，把下面格子的 `CONFIRM_PUBLISH` 改回 `True`，再运行「6.6 正式发布」。
（仍受 `DRY_RUN` 控制：`DRY_RUN=True` 只建草稿/只模拟，`False` 才真推送。）

In [ ]:
# ===== 发布确认开关 =====
CONFIRM_PUBLISH = False   # 👈 预览无误后，把这里改成 True

def confirm_and_publish(author=None, dry_run=DRY_RUN):
    """二次确认后发布：需 CONFIRM_PUBLISH=True 才真正调用。"""
    if not CONFIRM_PUBLISH:
        print("⛔ 尚未确认。请先仔细检查上方预览，再把 CONFIRM_PUBLISH 改为 True 后重跑此格。")
        return None
    if not LAST:
        print("⛔ 没有可发布的内容，请先运行『6 一键全流程』生成预览。")
        return None
    title = LAST["title"]; text = LAST["text"]; urls = LAST["urls"]
    names = LAST["names"]; html = LAST["html"]
    author = author if author is not None else LAST.get("author", "")
    print("③ 发文…")
    res = {}
    if WX_APPID and WX_SECRET:
        res["wechat"] = wechat_publish(title, html, names, WX_APPID, WX_SECRET, author, dry_run=dry_run)
        print("公众号:", res["wechat"])
    else:
        print("⚠️ 未配置 WX_APPID/WX_SECRET，跳过公众号。")
    if TT_KEY and TT_SECRET:
        res["toutiao"] = toutiao_publish(title, html, names, TT_KEY, TT_SECRET, dry_run=dry_run)
        print("头条号:", res["toutiao"])
    else:
        print("⚠️ 未配置 TT_KEY/TT_SECRET，跳过头条号。")
    return res

print("CONFIRM_PUBLISH =", CONFIRM_PUBLISH, "（改 True 后运行 6.6 发布）")

## 6.6 正式发布（确认后运行）

把 6.5 的 `CONFIRM_PUBLISH` 改为 `True` 后，运行下面这格即可发布。
建议：0 配置区 `DRY_RUN` 先 `True` 试跑（只建草稿不真发），确认对版再 `False` 真发。

In [ ]:
res = confirm_and_publish()
if res is not None:
    print("✅ 发布流程执行完毕，见上方各平台返回。")

## 7. 收尾：打包下载（不经过 10G 网盘）

`/content/output` 下所有图片打包成 zip，直接下载到你本机。

In [ ]:
import zipfile
from google.colab import files

zip_path = "/content/配图输出.zip"
with zipfile.ZipFile(zip_path, "w") as z:
    for root, _, fs in os.walk(OUT_DIR):
        for f in fs:
            z.write(os.path.join(root, f), f)
print("已打包：", zip_path)
files.download(zip_path)
print("✅ 完成：图片已生成、已上传图床、已打包下载到本地。")